In [1]:
#!/usr/bin/env python3
# tester_fleet_object_nav.py
# Rewrite: use the *adapted* fleet_object (ShipsActivity) for all logic checks.
# - Build fleet_object from get_my_ships() via adapters
# - Use ShipsActivity fields (fuel_level/status/current_waypoint) for decisions
# - When API calls mutate state (nav/refuel/orbit), merge the results back into fleet_object

from __future__ import annotations

from runtime_support import (
    maybe_await,
    unwrap_data,
    fmt_dt,
    now_utc,
    is_in_transit,
    status_value,
    api_get_my_ships,
    api_get_my_agent,
    api_get_ship_nav,
    api_refuel_ship,
    api_orbit_ship,
    api_dock_ship,
    api_navigate_ship,
    api_get_system_waypoints,
)

import asyncio
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, Optional

# Infra helpers (no DTOs leak into logic)
from runtime_support import setup_client_from_env, maybe_await, unwrap_data  # type: ignore

# Generated APIs
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi

# Domain + adapters
from domain.ships_activity import ShipsActivity
from adapters.ships_activity_adapter import (
    adapt_ships_activity_from_ship,
    merge_activity_with_nav,
)

# ----------------------------- util -------------------------------------------
def _fmt_dt(dt: Optional[datetime]) -> str:
    if not dt:
        return "None"
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat(timespec="seconds")

def _status_value(status_obj: Any) -> str:
    if status_obj is None:
        return ""
    val = getattr(status_obj, "value", None)
    if isinstance(val, str):
        return val
    name = getattr(status_obj, "name", None)
    if isinstance(name, str):
        return name
    s = str(status_obj)
    if "." in s:
        s = s.split(".")[-1]
    return s

def _is_in_transit_status(status_obj: Any) -> bool:
    return _status_value(status_obj) == "IN_TRANSIT"

def _build_nav_request(waypoint_symbol: str) -> Any:
    try:
        from openapi_client.models.navigate_ship_request import NavigateShipRequest  # type: ignore
        return NavigateShipRequest(waypoint_symbol=waypoint_symbol)
    except Exception:
        return {"waypoint_symbol": waypoint_symbol, "waypointSymbol": waypoint_symbol}



# ------------------------------ waiters ---------------------------------------
async def wait_while_in_transit(fleet_api: FleetApi, fleet_obj: FleetObject, ship_symbol: str) -> None:
    """Poll nav until not IN_TRANSIT; keep fleet_obj synced from nav."""
    while True:
        nav_resp = await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol)
        nav = unwrap_data(nav_resp)
        updated = fleet_obj.update_from_nav(ship_symbol, nav)
        if not _is_in_transit_status(updated.status):
            return
        arrival = getattr(getattr(nav, "route", None), "arrival", None)
        print(f"[WAIT] {ship_symbol} in transit; arrival {_fmt_dt(arrival)}")
        now = datetime.now(timezone.utc)
        if arrival and arrival.tzinfo is None:
            arrival = arrival.replace(tzinfo=timezone.utc)
        if arrival:
            secs = (arrival - now).total_seconds()
            await asyncio.sleep(max(secs, 0) + 1.0)
        else:
            await asyncio.sleep(2.0)

# --------------------------- prep & navigate ----------------------------------
async def ensure_ready_to_navigate(
    fleet_api: FleetApi,
    fleet_obj: FleetObject,
    ship_symbol: str,
    target_waypoint: str,
) -> None:
    """Use fleet_obj only for decisions; sync from nav/refuel responses."""
    # 1) Get latest nav and merge to fleet_obj
    nav_resp = await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol)
    nav_dto = unwrap_data(nav_resp)
    act = fleet_obj.update_from_nav(ship_symbol, nav_dto)
    print(f"[SNAP] {ship_symbol}: status={act.status}, wp={act.current_waypoint}, fuel_level={act.fuel_level}")

    # Already there and not in transit?
    if act.current_waypoint == target_waypoint and not _is_in_transit_status(act.status):
        print(f"[INFO] {ship_symbol} already at {target_waypoint} — no nav needed.")
        return

    # If in transit, wait
    if _is_in_transit_status(act.status):
        print(f"[INFO] {ship_symbol} IN_TRANSIT → waiting for arrival...")
        await wait_while_in_transit(fleet_api, fleet_obj, ship_symbol)
        act = fleet_obj.get(ship_symbol)  # refreshed by waiter

    # Fuel check via domain property
    needs_fuel = (act.fuel_level is None) or (act.fuel_level < 1.0)
    print(f"[CHECK] fuel_level={act.fuel_level} → needs_refuel={needs_fuel}")
    if needs_fuel:
        # Dock
        print(f"[ACTION] dock_ship({ship_symbol})")
        try:
            await maybe_await(fleet_api, "dock_ship", ship_symbol=ship_symbol)
        except Exception as e:
            print(f"[WARN] dock_ship failed: {e}")

        # Refuel (empty JSON body to avoid 422)
        print(f"[ACTION] refuel_ship({ship_symbol})")
        refuel_resp = None
        try:
            refuel_resp = await maybe_await(
                fleet_api, "refuel_ship", ship_symbol=ship_symbol, refuel_ship_request={}
            )
        except TypeError:
            try:
                refuel_resp = await maybe_await(
                    fleet_api, "refuel_ship", ship_symbol=ship_symbol, body={}
                )
            except Exception as e:
                print(f"[WARN] refuel_ship failed: {e}")
        except Exception as e:
            print(f"[WARN] refuel_ship failed: {e}")

        if refuel_resp is not None:
            fleet_obj.update_from_refuel(ship_symbol, unwrap_data(refuel_resp))
            print(f"[SNAP] post-refuel fuel_level={fleet_obj.get(ship_symbol).fuel_level}")

        # Orbit back (if not in transit)
        nav_chk = unwrap_data(await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol))
        act = fleet_obj.update_from_nav(ship_symbol, nav_chk)
        if not _is_in_transit_status(act.status):
            print(f"[ACTION] orbit_ship({ship_symbol})")
            try:
                await maybe_await(fleet_api, "orbit_ship", ship_symbol=ship_symbol)
                # refresh activity after orbit
                nav_chk2 = unwrap_data(await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol))
                fleet_obj.update_from_nav(ship_symbol, nav_chk2)
            except Exception as e:
                print(f"[WARN] orbit_ship failed: {e}")

    # Ensure IN_ORBIT before navigating
    nav_final = unwrap_data(await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol))
    act = fleet_obj.update_from_nav(ship_symbol, nav_final)
    if _status_value(act.status) != "IN_ORBIT":
        print(f"[ACTION] ensure orbit (status={act.status})")
        try:
            await maybe_await(fleet_api, "orbit_ship", ship_symbol=ship_symbol)
            nav_final2 = unwrap_data(await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol))
            fleet_obj.update_from_nav(ship_symbol, nav_final2)
        except Exception as e:
            print(f"[WARN] orbit_ship failed: {e}")

    act = fleet_obj.get(ship_symbol)
    print(f"[READY] {ship_symbol}: status={act.status}, wp={act.current_waypoint}, fuel_level={act.fuel_level}")

async def navigate_and_wait(fleet_api: FleetApi, fleet_obj: FleetObject, ship_symbol: str, target_wp: str) -> None:
    # Skip if already there & not in transit
    act = fleet_obj.get(ship_symbol)
    if act and act.current_waypoint == target_wp and not _is_in_transit_status(act.status):
        print(f"[SKIP] {ship_symbol} already at {target_wp}")
        return

    print(f"[NAV] request: {ship_symbol} → {target_wp}")
    req = _build_nav_request(target_wp)
    try:
        nav_resp = await maybe_await(fleet_api, "navigate_ship", ship_symbol=ship_symbol, navigate_ship_request=req)
    except TypeError:
        nav_resp = await maybe_await(fleet_api, "navigate_ship", ship_symbol=ship_symbol, body=req)

    nav_dto = unwrap_data(nav_resp)
    act = fleet_obj.update_from_nav(ship_symbol, nav_dto)
    arrival = act.arr_time
    dest = act.destination_waypoint or target_wp
    print(f"[NAV] accepted: {ship_symbol} → {dest}, arrival {_fmt_dt(arrival)}")

    await wait_while_in_transit(fleet_api, fleet_obj, ship_symbol)
    act = fleet_obj.get(ship_symbol)
    print(f"[NAV] arrived: {ship_symbol} at {act.current_waypoint}")


In [2]:

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)

from runtime_support import build_fleet_object

# Build adapted fleet_object (no DTOs in business logic)
ships_activity_obj = await build_fleet_object(fleet_api)

# Select ship or fall back
act = ships_activity_obj.get("TROOTS-1")
#print("INFO: this is the TROOTS-1 ship data from fleet_obj", act, "END INFO fleet_obj")
#INFO: this is the TROOTS-1 ship data from fleet_obj 
# symbol='TROOTS-1' 
# status='IN_ORBIT' 
# dep_time=datetime.datetime(2025, 9, 9, 10, 38, 22, 749000, tzinfo=TzInfo(UTC)) 
# arr_time=datetime.datetime(2025, 9, 9, 10, 38, 48, 749000, tzinfo=TzInfo(UTC)) 
# flight_mode='CRUISE' 
# cooldown_remaining_seconds=0 
# cooldown_expiration=None 
# cargo_units=3 
# cargo_capacity=40 
# fuel_current=400 
# fuel_capacity=400 
# current_waypoint='X1-Q51-A1' 
# destination_waypoint='X1-Q51-A1' 
# condition=0.999

arr_secs = act.arr_time
now = datetime.now(timezone.utc)
arr_time = (arr_secs - now).total_seconds()
print("Arrival seconds: ", arr_time)

print("INFO: this is the status from TROOTS-1, as called by fleet_obj.fuel_level: ", act.fuel_level)
print("INFO: this is the status from TROOTS-1, as called by fleet_obj.status: ", act.status)

print("This is the return from act.in_transit_check: ", act.transit_check)
print("This is the return from act.refuel_check: ", act.refuel_check)
print("This is the return from act.in_orbit_check: ", act.in_orbit_check)

print("This is the return of act.symbol: ", act.symbol)

await api_refuel_ship(fleet_api, act.symbol)



[BOOT] Adapted 2 ships into fleet_object
Arrival seconds:  -394.866329
INFO: this is the status from TROOTS-1, as called by fleet_obj.fuel_level:  0.765
INFO: this is the status from TROOTS-1, as called by fleet_obj.status:  IN_ORBIT
Not in transit, READY
This is the return from act.in_transit_check:  False
refueling
This is the return from act.refuel_check:  True
In orbit, READY TO NAVIGATE
This is the return from act.in_orbit_check:  True
This is the return of act.symbol:  TROOTS-1


In [ ]:
import asyncio
import inspect

def build_nav_request(waypoint_symbol: str) -> Any:
    try:
        from openapi_client.models.navigate_ship_request import NavigateShipRequest  # type: ignore
        return NavigateShipRequest(waypoint_symbol=waypoint_symbol)
    except Exception:
        return {"waypoint_symbol": waypoint_symbol, "waypointSymbol": waypoint_symbol}
    
async def maybe_await(api_obj: Any, method_name: str, *args, **kwargs) -> Any:
    fn = getattr(api_obj, method_name)
    if inspect.iscoroutinefunction(fn):
        return await fn(*args, **kwargs)
    result = fn(*args, **kwargs)
    if inspect.isawaitable(result):
        return await result
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(None, lambda: fn(*args, **kwargs))

def unwrap_data(resp: Any) -> Any:
    return getattr(resp, "data", resp)

async def api_orbit_ship(fleet_api, ship_symbol: str) -> None:
    await maybe_await(fleet_api, "orbit_ship", ship_symbol=ship_symbol)

async def api_dock_ship(fleet_api, ship_symbol: str) -> None:
    await maybe_await(fleet_api, "dock_ship", ship_symbol=ship_symbol)

async def api_get_ship_nav(fleet_api, ship_symbol: str) -> Any:
    resp = await maybe_await(fleet_api, "get_ship_nav", ship_symbol=ship_symbol)
    return unwrap_data(resp)

NameError: name 'log_intent' is not defined

In [ ]:
from datetime import timezone
import datetime

async def api_navigate_ship(fleet_api, ship_symbol: str, waypoint_symbol: str) -> Any:
    print("starting navigation")
    print("STEP 1: nav prep")
    ready = await nav_prep(fleet_api, ship_symbol=ship_symbol, waypoint_symbol=waypoint_symbol)
    print("Prep complete")
    if not ready:
        print("[SKIP] Navigation aborted, already at destination")
        return None  # stop here
    print("Step 2: build_nav_request")
    req = build_nav_request(waypoint_symbol)
    print("This is the build_nav_request outpt: ", req)
    print("build_nav_request done")
    print("Step 3: await navigate_ship")
    resp = await maybe_await(fleet_api, "navigate_ship", ship_symbol=ship_symbol, navigate_ship_request=req)
    return unwrap_data(resp)


async def nav_prep(fleet_api, ship_symbol: str, waypoint_symbol: str) -> bool:
    ships_activity_obj = await build_fleet_object(fleet_api)
    act = ships_activity_obj.get(ship_symbol)

    # already at destination → stop
    if act.destination_waypoint == waypoint_symbol:
        print("Ship is already at the destination")
        return False
    else:
        print("Not at destination, continuing")

    if act.transit_check:
        now = datetime.datetime.now(datetime.timezone.utc)
        secs_to_arrival = (act.arr_time - now).total_seconds()
        print("Seconds until arrival:", secs_to_arrival)
        await asyncio.sleep(secs_to_arrival + 3)
        print("Arrived and ready")
        update = await build_fleet_object(fleet_api)
        new_status = update.get(act.symbol).status
        act.status = new_status
        print("After arriving, the status of", act.symbol, "is", act.status)

    if act.refuel_check:
        print("Refuelling now...")
        await api_dock_ship(fleet_api, act.symbol)
        act.status = "DOCKED"
        await asyncio.sleep(3)

    if not act.in_orbit_check:
        print("Not in orbit, going into orbit now...")
        await api_orbit_ship(fleet_api, act.symbol)
        await asyncio.sleep(3)

    return True   # ready to navigate



In [ ]:
nav_resp = await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H52")
#print("This is nav_resp: ", nav_resp)
print("Sleeping for 4 seconds")
await asyncio.sleep(4)
nav_resp2 = await api_get_ship_nav(fleet_api, "TROOTS-1")
print("This is the current ship status: ", nav_resp2.status)


#await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H52")


starting navigation
STEP 1: nav prep
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Not in transit, READY
refueling
Refuelling now...
Not in orbit, going into orbit now
Not in orbit, going into orbit now...
Prep complete
Step 2: build_nav_request
build_navv_request done
Step 3: await navigate_ship


BadRequestException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Access-Control-Allow-Origin': '*', 'Access-Control-Expose-Headers': 'Retry-After, X-RateLimit-Type, X-RateLimit-Limit-Burst, X-RateLimit-Limit-Per-Second, X-RateLimit-Remaining, X-RateLimit-Reset', 'Content-Length': '349', 'Content-Type': 'application/json; charset=utf-8', 'Date': 'Thu, 11 Sep 2025 07:21:41 GMT', 'Retry-After': '1', 'X-Ratelimit-Limit-Burst': '30', 'X-Ratelimit-Limit-Per-Second': '2', 'X-Ratelimit-Remaining': '0', 'X-Ratelimit-Reset': '2025-09-11T07:21:43.067Z', 'X-Ratelimit-Type': 'IP Address'})
HTTP response body: {"error":{"code":4214,"message":"Ship is currently in-transit from X1-Q51-AB5A to X1-Q51-H52 and arrives in 42 seconds.","data":{"departureSymbol":"X1-Q51-AB5A","destinationSymbol":"X1-Q51-H52","arrival":"2025-09-11T07:22:25.081Z","departureTime":"2025-09-11T07:21:42.081Z","secondsToArrival":42},"requestId":"019937a6-cca6-72d7-b191-3356874452d0"}}


In [ ]:
# Result of refuel:  
# agent=Agent
# (account_id='cmeb98xyw0028tm167bfnq149', 
# symbol='TROOTS', 
# headquarters='X1-Q51-A1', 
# credits=173488, 
# starting_faction='AEGIS', 
# ship_count=2) 
# fuel=ShipFuel
# (current=400, capacity=400, 
# consumed=ShipFuelConsumed(
# amount=53, 
# timestamp=datetime.datetime(2025, 9, 9, 18, 55, 27, 489000, tzinfo=TzInfo(UTC)))) 
# cargo=None transaction=MarketTransaction(waypoint_symbol='X1-Q51-A2', 
# ship_symbol='TROOTS-1', trade_symbol='FUEL', type='PURCHASE', units=0, price_per_unit=72, total_price=0, timestamp=datetime.datetime(2025, 9, 9, 19, 9, 3, 200000, tzinfo=TzInfo(UTC)))

[INFO] Agent HQ waypoint: X1-Q51-A1


NameError: name 'ship_symbol' is not defined